# 10v3 — Host Mode-A: Mirror/Near-Duplicate Symmetry + Freeze Attempt

**Task 1 of a two-task gated milestone** (Task 2, Notebook 11, may not start unless this
notebook writes a machine-readable `FROZEN` status). This notebook does NOT reach `FROZEN` --
it honestly reports `PROVISIONAL` with the specific blocking items enumerated in section "Mode-A
freeze decision" below, and per the milestone's own hard-stop rule, **Task 2 is not started this
session**.

What this notebook actually adds over `10v2`:

1. **Mirror vs. near-duplicate distinction** (`casmi.spectra.deduplication.
   classify_reference_relationship`) -- `mirror` (exact `peak_hash` match: the same physical
   spectrum ingested twice) is now tracked SEPARATELY from `near_duplicate_non_mirror`
   (spectrally similar but not byte-identical -- a legitimate independent remeasurement).
   Conflating them (as `10v2`'s single `duplicate_aware` protocol did) would treat real
   independent evidence as if it were a database artifact.
2. **Symmetric classification**: `10v2` only checked TRUE candidates' references for
   near-duplicates (reasoning that decoys are a different molecule so it seemed physically
   implausible) -- this notebook checks a bounded, ranked reference pool for **every**
   (query, candidate) row, true or decoy, because a `mirror` specifically CAN occur for a decoy:
   it indicates the same physical spectrum was ingested under two different structure labels
   somewhere in the library, a real data-quality signal regardless of which candidate it lands on.
3. **`mirror_aware`** (excludes only `mirror`) is now the PRIMARY protocol, distinct from
   `near_dup_strict` (excludes `mirror` AND `near_duplicate_non_mirror`, kept as a sensitivity
   analysis -- high spectral similarity alone can be legitimate evidence, not just a database
   artifact).
4. **Formalized, persisted cap-bug audit** (`host_reference_cap_audit.parquet`) reproducing the
   OLD (file-order-capped) selection logic as an isolated diagnostic function and comparing it
   directly against the corrected selection, with `compat_rank` distribution reporting.
5. **Decision log** (`outputs/decision_log.jsonl`, `casmi.validation.decision_log`) -- every
   choice that could affect the host number gets one recorded entry (`bug_fix` /
   `pre_registered_choice` / `post_host_change`), so "did we tune on the host?" is answerable by
   reading a file.

**Explicitly NOT implemented this notebook** (blocking `FROZEN`, honestly enumerated at the
end): protocol-matched DEV retraining (ranker variants V0-V4, including source-held-out and
source+structure-held-out), the timsTOF mass log-likelihood model, transparent mass+spectral
fusion baselines (B1-B5), connectivity-cluster bootstrap (current CI is still spectrum-level),
candidate-density analysis, near-miss analysis. These are each substantial, separable pieces of
work -- attempting all of them in the same pass as the mirror/symmetry fix would risk doing
each one shallowly rather than correctly.

In [1]:
import sys, json, time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import lightgbm as lgb

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact, artifact_fingerprint, save_artifact
from casmi.io.parquet import load_rows_by_offset
from casmi.candidates.cache import cached_mass_index
from casmi.candidates.mass_index import MassIndex
from casmi.candidates.generator import CandidateGenerator, dedupe_pool_to_connectivity
from casmi.spectra.preprocessing import remove_invalid_peaks, truncate_top_peaks
from casmi.spectra.library import build_reference_index
from casmi.spectra.deduplication import classify_reference_relationship, compute_peak_hash
from casmi.ranking.features import compute_pair_spectral_scores
from casmi.ranking.feature_policy import SPECTRAL_RANKING_FEATURE_POLICY, assert_no_prohibited_features
from casmi.ranking.evaluation import rank_candidates, ranking_metrics, true_candidate_rank, per_query_reciprocal_rank
from casmi.validation.checks import CheckResult, summarize_checks
from casmi.validation.baseline_registry import load_baseline, load_promoted_baseline, register_baseline
from casmi.validation.bootstrap import paired_bootstrap_report
from casmi.validation.decision_log import log_decision, load_decision_log

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
N_BOOTSTRAP = 5000
HOST_INGEST_LIB = "enveda-np-examples"
MAX_REFS_PER_CANDIDATE = 5
WALK_CAP = 15  # generous bounded pool per row (both true AND decoy) -- see section 7.3 note below
BIN_WIDTH_DA = 0.1
PEAK_TOL_DA = 0.02
MAX_PEAKS_PER_SPECTRUM = 100
DEDUP_COSINE_THRESHOLD = 0.95
DEDUP_PRECURSOR_DIFF_DA = 0.01
LGBM_PARAMS = dict(
    objective="lambdarank", metric="ndcg", n_estimators=300, learning_rate=0.05,
    num_leaves=31, min_child_samples=20, random_state=SEED, verbosity=-1,
)

paths = get_project_paths()
CG_INTERIM = paths.interim / "candidate_generation"
CG_PROCESSED = paths.processed / "candidate_generation"
RANK_INTERIM = paths.interim / "spectral_ranking"
HOST_INTERIM = paths.interim / "host_holdout"
HOST_PROCESSED = paths.processed / "host_holdout"
BASELINE_REGISTRY_PATH = paths.processed / "baseline_registry.json"
REPORTS_DIR = paths.reports
DECISION_LOG_PATH = paths.outputs / "decision_log.jsonl"

MODEL_FEATURES = SPECTRAL_RANKING_FEATURE_POLICY.model_features
checks = []
print(f"MODEL_FEATURES: {MODEL_FEATURES}")

# retroactively record the v1->v2 cap fix as a bug_fix, not model tuning (spec section 5)
log_decision(
    decision="Fixed v1's file-order pre-cap (first 30 refs) before protocol filtering -> v2's filter-then-rank-then-cap",
    evidence_used="Ad hoc audit (audit_cap_bug.py): 0/126,573 rows lost eligible-reference coverage below MAX_REFS=5 because of the cap, but 94% of TRUE candidates had >30 total references and got an arbitrary (not compatibility-ranked) subset selected",
    host_visible=True, category="bug_fix", log_path=DECISION_LOG_PATH,
)
print(f"decision log: {len(load_decision_log(DECISION_LOG_PATH))} entries")

MODEL_FEATURES: ['abs_mass_error_ppm', 'cosine_max', 'cosine_top3_mean', 'modified_cosine_max', 'modified_cosine_top3_mean', 'peak_overlap_frac_max', 'peak_overlap_frac_top3_mean', 'neutral_loss_cosine_max', 'neutral_loss_cosine_top3_mean']
decision log: 3 entries


## 1. Reload host holdout artifacts + rebuild candidate pool (unaffected by any of this notebook's changes)

In [2]:
host_holdout_spectra = pd.read_parquet(HOST_PROCESSED / "host_holdout_spectra.parquet")
all_host_query_ids = host_holdout_spectra["query_id"].tolist()

train_meta = load_artifact("train_spectrum_metadata")
with open(CG_PROCESSED / "candidate_generation_contract.json") as f:
    contract = json.load(f)
SELECTED_PPM = contract["molecule_policy"]["tolerance_ppm"]

molecule_mass_variants = pd.read_parquet(CG_INTERIM / "molecule_mass_variants.parquet")
library_fp = artifact_fingerprint("molecule_mass_variants", CG_INTERIM)
mass_index, _, _ = cached_mass_index(
    CG_INTERIM, build_fn=lambda: MassIndex.from_dataframe(molecule_mass_variants, mass_col="exact_mass", key_col="mass_variant_id"),
    library_fingerprint=library_fp, mass_col="exact_mass", key_col="mass_variant_id",
)
variant_to_connectivity = molecule_mass_variants.set_index("mass_variant_id")["connectivity_key"]
gen = CandidateGenerator(mass_index)

reference_index = build_reference_index(train_meta, connectivity_col="connectivity_key", id_col="train_spectrum_id")
spectrum_id_to_ingest_lib = train_meta.set_index("train_spectrum_id")["ingest_lib"].to_dict()
spectrum_id_to_adduct = train_meta.set_index("train_spectrum_id")["adduct"].to_dict()
spectrum_id_to_ce_mean = train_meta.set_index("train_spectrum_id")["ce_mean"].to_dict()
spectrum_id_to_instrument = train_meta.set_index("train_spectrum_id")["instrument_type"].to_dict()

host_pool_raw = gen.generate_dataframe(
    host_holdout_spectra, tolerance_ppm=SELECTED_PPM,
    query_id_col="query_id", mass_col="neutral_mass", true_key_col="true_connectivity_key",
)
host_pool = dedupe_pool_to_connectivity(host_pool_raw, variant_to_connectivity)
print(f"host_pool: {len(host_pool):,} (query, candidate) pairs over {host_pool['query_id'].nunique():,} / {len(host_holdout_spectra):,} host queries")
checks.append(CheckResult("host holdout + candidate pool reloaded", len(host_pool) > 0, f"{len(host_pool):,} pairs"))

[CACHE HIT] mass_index <- C:\Users\myben\OneDrive\Documents\Enveda\data\interim\candidate_generation\mass_index.npz (library_fingerprint matches)


host_pool: 126,573 (query, candidate) pairs over 1,184 / 1,184 host queries


## 2. Section 7.2 — deterministic compatibility ranking (metadata only)

`(same_adduct desc, same_ionization desc, ce_diff asc [unknown CE sorts last], same_instrument
desc, ref_spectrum_id asc)`. No spectral computation needed for this step -- pure dict lookups
over the FULL reference list, no pre-filter cap anywhere (section 7.1).

In [3]:
def offset_of(spectrum_id):
    return int(spectrum_id.split("_")[1])


spectrum_id_to_ionization = train_meta.set_index("train_spectrum_id")["ionization_mode"].to_dict()
query_meta = host_holdout_spectra.set_index("query_id")[["adduct", "ce_mean", "ionization_mode", "instrument_type", "true_connectivity_key"]]


def compat_sort_key(rid, q_adduct, q_ce, q_ion, q_instrument):
    same_adduct = spectrum_id_to_adduct.get(rid) == q_adduct
    same_ion = spectrum_id_to_ionization.get(rid) == q_ion
    same_instrument = spectrum_id_to_instrument.get(rid) == q_instrument
    ce = spectrum_id_to_ce_mean.get(rid)
    ce_diff = abs(ce - q_ce) if (pd.notna(ce) and pd.notna(q_ce)) else float("inf")
    return (0 if same_adduct else 1, 0 if same_ion else 1, ce_diff, 0 if same_instrument else 1, rid)


def ranked_full_reference_list(candidate_key, query_id, q_adduct, q_ce, q_ion, q_instrument):
    """FULL eligible (self-excluded) reference list for `candidate_key`, ranked by compat_rank.
    NO truncation here -- section 7.1's requirement. Truncation happens only in section 3, AFTER
    this full ranking, and only as the (generous, symmetric) WALK_CAP bound."""
    full = [rid for rid in reference_index.get(candidate_key, ()) if rid != query_id]
    full.sort(key=lambda rid: compat_sort_key(rid, q_adduct, q_ce, q_ion, q_instrument))
    return full


checks.append(CheckResult("deterministic compat-rank reference ordering implemented (no pre-filter cap)", True,
                           "same_adduct, same_ionization, |CE diff| (unknown CE last), same_instrument, stable id"))

## 3. Section 7.3 — bounded, SYMMETRIC walk pool (true AND decoy candidates alike)

`10v2` only checked near-duplicates for TRUE-candidate rows (reasoning a decoy is a different
molecule, so a near-*duplicate* seemed implausible) -- but a `mirror` specifically is a
data-provenance issue (same physical spectrum, two labels) that CAN occur regardless of which
candidate it lands on. This walks the top `WALK_CAP` compat-ranked references for **every**
(query, candidate) row and classifies each one (section 4) -- a generous bounded approximation
of literal per-reference lazy early-stopping (section 7.3's pseudocode), chosen because state-
tracking 4 simultaneous per-protocol stopping conditions one reference at a time is not
meaningfully more correct here (near-duplicate/mirror rates are low enough that `WALK_CAP=15`
almost always provides more than `MAX_REFS=5` clean references per protocol -- verified in
section 6's coverage audit) while being far simpler to implement correctly and to reason about.

In [4]:
t0 = time.time()
row_walk_pool = {}  # (query_id, candidate_key) -> ranked list of up to WALK_CAP reference ids
for row in host_pool.itertuples(index=False):
    qmeta = query_meta.loc[row.query_id]
    full_ranked = ranked_full_reference_list(row.candidate_connectivity_key, row.query_id,
                                              qmeta["adduct"], qmeta["ce_mean"], qmeta["ionization_mode"], qmeta["instrument_type"])
    row_walk_pool[(row.query_id, row.candidate_connectivity_key)] = full_ranked[:WALK_CAP]

print(f"walk pools built for {len(row_walk_pool):,} rows (symmetric -- true and decoy alike) in {time.time() - t0:.1f}s")
checks.append(CheckResult("symmetric walk pool built for every (query, candidate) row", len(row_walk_pool) == len(host_pool), ""))

walk pools built for 126,573 rows (symmetric -- true and decoy alike) in 55.9s


## 4. Load peaks (ONE combined call) + classify every walked reference (mirror / near_duplicate_non_mirror / other)

In [5]:
needed_ref_ids = set()
for pool in row_walk_pool.values():
    needed_ref_ids.update(pool)
host_query_offsets = set(host_holdout_spectra["query_id"].apply(offset_of))
host_ref_offsets = set(offset_of(i) for i in needed_ref_ids)
host_all_offsets = host_query_offsets | host_ref_offsets
print(f"host: {len(host_query_offsets):,} query offsets + {len(host_ref_offsets):,} reference offsets = {len(host_all_offsets):,} total")

t0 = time.time()
raw = load_rows_by_offset(paths.train, ["ms2_mzs", "ms2_normalized_intensities", "precursor_mz"], list(host_all_offsets))
raw = raw.rename(columns={"_row_offset": "row_offset"})
host_peaks_by_offset = {}
for r in raw.itertuples(index=False):
    mzs, ints = remove_invalid_peaks(r.ms2_mzs, r.ms2_normalized_intensities)
    mzs, ints = truncate_top_peaks(mzs, ints, max_peaks=MAX_PEAKS_PER_SPECTRUM)
    host_peaks_by_offset[r.row_offset] = {"mzs": mzs, "intensities": ints, "precursor_mz": r.precursor_mz}
print(f"loaded {len(host_peaks_by_offset):,} host-side peak rows in {time.time() - t0:.1f}s")
checks.append(CheckResult("host peak arrays loaded (symmetric walk pool)", len(host_peaks_by_offset) > 0, f"{len(host_peaks_by_offset):,} spectra"))

host: 1,184 query offsets + 204,973 reference offsets = 205,347 total


loaded 205,347 host-side peak rows in 29.9s


In [6]:
# precompute peak_hash ONCE per unique spectrum offset -- classify_reference_relationship would
# otherwise re-hash the SAME query spectrum from scratch for every one of its (up to WALK_CAP)
# reference comparisons, and re-hash a popular reference every time a different query happens to
# walk past it. Sha256-ing a canonicalized peak string is cheap per call, but not free at
# millions of calls.
t0 = time.time()
offset_to_peak_hash = {
    off: compute_peak_hash(p["mzs"], p["intensities"], p["precursor_mz"])
    for off, p in host_peaks_by_offset.items()
}
print(f"precomputed {len(offset_to_peak_hash):,} peak hashes in {time.time() - t0:.1f}s")

t0 = time.time()
classification_rows = []
row_classified_refs = {}  # (query_id, candidate_key) -> [(ref_id, relationship), ...] in walk order

for row in host_pool.itertuples(index=False):
    key = (row.query_id, row.candidate_connectivity_key)
    q_offset = offset_of(row.query_id)
    if q_offset not in host_peaks_by_offset:
        row_classified_refs[key] = []
        continue
    q_peaks = host_peaks_by_offset[q_offset]
    q_hash = offset_to_peak_hash[q_offset]
    classified = []
    for rid in row_walk_pool[key]:
        r_offset = offset_of(rid)
        if r_offset not in host_peaks_by_offset:
            continue
        relationship = classify_reference_relationship(q_peaks, host_peaks_by_offset[r_offset],
                                                          cosine_threshold=DEDUP_COSINE_THRESHOLD, precursor_diff_da=DEDUP_PRECURSOR_DIFF_DA,
                                                          query_peak_hash=q_hash, reference_peak_hash=offset_to_peak_hash[r_offset])
        classified.append((rid, relationship))
        classification_rows.append({
            "query_id": row.query_id, "candidate_connectivity_key": row.candidate_connectivity_key,
            "is_true_candidate": bool(row.is_true_candidate), "reference_spectrum_id": rid,
            "reference_ingest_lib": spectrum_id_to_ingest_lib.get(rid),
            "is_same_source": spectrum_id_to_ingest_lib.get(rid) == HOST_INGEST_LIB,
            "relationship": relationship,
        })
    row_classified_refs[key] = classified

print(f"classified {len(classification_rows):,} (query, candidate, reference) triples in {time.time() - t0:.1f}s")
reference_classification = pd.DataFrame(classification_rows)
save_artifact(reference_classification, "host_reference_classification", HOST_PROCESSED,
              description="Section 4 (v3): symmetric mirror/near-duplicate classification for every walked reference, true AND decoy candidates alike")
checks.append(CheckResult("symmetric mirror/near-duplicate classification computed", len(reference_classification) > 0, f"{len(reference_classification):,} triples"))

precomputed 205,347 peak hashes in 23.2s


classified 900,492 (query, candidate, reference) triples in 25.0s


## 5. Section 11 — mirror/near-duplicate breakdown by source, TRUE vs decoy

In [7]:
dup_breakdown = reference_classification[reference_classification["relationship"] != "other"]
breakdown_summary = dup_breakdown.groupby(["relationship", "is_same_source", "is_true_candidate"]).size().rename("count").reset_index()
display(breakdown_summary)

n_same_source_mirror = int(((dup_breakdown["relationship"] == "mirror") & dup_breakdown["is_same_source"]).sum())
n_cross_source_mirror = int(((dup_breakdown["relationship"] == "mirror") & ~dup_breakdown["is_same_source"]).sum())
n_same_source_near_dup = int(((dup_breakdown["relationship"] == "near_duplicate_non_mirror") & dup_breakdown["is_same_source"]).sum())
n_cross_source_near_dup = int(((dup_breakdown["relationship"] == "near_duplicate_non_mirror") & ~dup_breakdown["is_same_source"]).sum())

queries_with_cross_source_mirror = reference_classification.loc[
    (reference_classification["relationship"] == "mirror") & ~reference_classification["is_same_source"], "query_id"
].nunique()
queries_with_cross_source_near_dup = reference_classification.loc[
    (reference_classification["relationship"] == "near_duplicate_non_mirror") & ~reference_classification["is_same_source"], "query_id"
].nunique()

# decoy mirrors specifically -- the symmetry fix's whole point: can a mirror land on a WRONG candidate?
n_decoy_mirrors = int(((dup_breakdown["relationship"] == "mirror") & ~dup_breakdown["is_true_candidate"]).sum())

print(f"same-source mirror: {n_same_source_mirror}, cross-source mirror: {n_cross_source_mirror}")
print(f"same-source near_duplicate_non_mirror: {n_same_source_near_dup}, cross-source: {n_cross_source_near_dup}")
print(f"host queries with >=1 cross-source mirror: {queries_with_cross_source_mirror} / {len(all_host_query_ids)} ({queries_with_cross_source_mirror / len(all_host_query_ids):.2%})")
print(f"host queries with >=1 cross-source near_duplicate_non_mirror: {queries_with_cross_source_near_dup} / {len(all_host_query_ids)} ({queries_with_cross_source_near_dup / len(all_host_query_ids):.2%})")
print(f"\nDECOY-candidate mirrors found: {n_decoy_mirrors} -- {'symmetry fix caught real cases a true-only check would have missed' if n_decoy_mirrors > 0 else 'none observed in this holdout, but the check is now structurally symmetric regardless'}")

save_artifact(breakdown_summary, "host_duplicate_mirror_breakdown", HOST_PROCESSED, description="Section 5 (v3): mirror/near-dup counts by source and true/decoy status")
checks.append(CheckResult("mirror/near-duplicate breakdown computed, symmetric true/decoy", True, f"decoy mirrors: {n_decoy_mirrors}"))

,relationship,is_same_source,is_true_candidate,count
0,near_duplicate_non_mirror,False,False,1105
1,near_duplicate_non_mirror,False,True,1578
2,near_duplicate_non_mirror,True,False,4
3,near_duplicate_non_mirror,True,True,144


same-source mirror: 0, cross-source mirror: 0
same-source near_duplicate_non_mirror: 148, cross-source: 2683
host queries with >=1 cross-source mirror: 0 / 1184 (0.00%)
host queries with >=1 cross-source near_duplicate_non_mirror: 423 / 1184 (35.73%)

DECOY-candidate mirrors found: 0 -- none observed in this holdout, but the check is now structurally symmetric regardless


## 6. Section 10 — reference availability audit, TRUE vs decoy, per protocol

In [8]:
def protocol_eligible(classified_refs, protocol):
    if protocol == "standard":
        return [rid for rid, rel in classified_refs]
    if protocol == "cross_library":
        return [rid for rid, rel in classified_refs if spectrum_id_to_ingest_lib.get(rid) != HOST_INGEST_LIB]
    if protocol == "mirror_aware":
        return [rid for rid, rel in classified_refs if spectrum_id_to_ingest_lib.get(rid) != HOST_INGEST_LIB and rel != "mirror"]
    if protocol == "near_dup_strict":
        return [rid for rid, rel in classified_refs if spectrum_id_to_ingest_lib.get(rid) != HOST_INGEST_LIB and rel not in ("mirror", "near_duplicate_non_mirror")]
    raise ValueError(protocol)


PROTOCOLS = ("standard", "cross_library", "mirror_aware", "near_dup_strict")
PRIMARY_PROTOCOL = "mirror_aware"

availability_rows = []
for row in host_pool.itertuples(index=False):
    key = (row.query_id, row.candidate_connectivity_key)
    classified = row_classified_refs[key]
    for protocol in PROTOCOLS:
        n_elig = len(protocol_eligible(classified, protocol))
        availability_rows.append({"query_id": row.query_id, "candidate_connectivity_key": row.candidate_connectivity_key,
                                    "is_true_candidate": bool(row.is_true_candidate), "protocol": protocol, "n_eligible": n_elig})
availability_df = pd.DataFrame(availability_rows)

coverage_summary = []
for protocol in PROTOCOLS:
    for group_name, is_true in [("true_candidates", True), ("decoys", False)]:
        sub = availability_df[(availability_df["protocol"] == protocol) & (availability_df["is_true_candidate"] == is_true)]
        coverage_summary.append({
            "protocol": protocol, "group": group_name,
            "frac_ge1_ref": (sub["n_eligible"] >= 1).mean(),
            "frac_ge3_refs": (sub["n_eligible"] >= 3).mean(),
            "frac_ge5_refs": (sub["n_eligible"] >= MAX_REFS_PER_CANDIDATE).mean(),
            "n_zero_refs": int((sub["n_eligible"] == 0).sum()),
            "n_rows": len(sub),
        })
reference_coverage = pd.DataFrame(coverage_summary)
display(reference_coverage)

primary_true_zero = reference_coverage.query("protocol == @PRIMARY_PROTOCOL and group == 'true_candidates'")["n_zero_refs"].iloc[0]
zero_ref_fraction = primary_true_zero / len(all_host_query_ids)
print(f"\n{PRIMARY_PROTOCOL}: {primary_true_zero} / {len(all_host_query_ids)} true candidates ({zero_ref_fraction:.2%}) have ZERO eligible references -- REFERENCE_UNAVAILABLE_SUBGROUP")
if zero_ref_fraction > 0.05:
    print("*** >5% -- report this subgroup separately, do not treat host as one homogeneous Class-1 set ***")
else:
    print("(below the 5% threshold)")

save_artifact(availability_df, "host_reference_coverage", HOST_PROCESSED, description="Section 6 (v3): per-row, per-protocol eligible-reference counts, true/decoy split")
checks.append(CheckResult("reference availability audit computed, true/decoy split", True, f"{PRIMARY_PROTOCOL} zero-ref true candidates: {primary_true_zero}"))

,protocol,group,frac_ge1_ref,frac_ge3_refs,frac_ge5_refs,n_zero_refs,n_rows
0,standard,true_candidates,1.000000,1.000000,0.997466,0,1184
1,standard,decoys,1.000000,0.916181,0.630271,0,125389
2,cross_library,true_candidates,1.000000,0.993243,0.989865,0,1184
3,cross_library,decoys,1.000000,0.916181,0.630271,0,125389
4,mirror_aware,true_candidates,1.000000,0.993243,0.989865,0,1184
5,mirror_aware,decoys,1.000000,0.916181,0.630271,0,125389
6,near_dup_strict,true_candidates,0.997466,0.978885,0.961149,3,1184
7,near_dup_strict,decoys,0.999912,0.915997,0.630095,11,125389



mirror_aware: 0 / 1184 true candidates (0.00%) have ZERO eligible references -- REFERENCE_UNAVAILABLE_SUBGROUP
(below the 5% threshold)


## 7. Section 9 — reproduce the OLD (file-order-capped) selection as an isolated diagnostic, audit its effect

Section 9.1: what fraction of the CORRECTED (compat_rank) selection actually came from beyond
where the old naive 30-item file-order cap would have looked?

In [9]:
OLD_CAP = 30


def old_cap_selection(candidate_key, query_id, protocol):
    """Exact reproduction of v1's buggy logic: take the first OLD_CAP references in raw
    reference_index (file/ingestion) order, THEN filter by protocol, THEN take MAX_REFS."""
    file_order = [rid for rid in reference_index.get(candidate_key, ()) if rid != query_id][:OLD_CAP]
    if protocol == "cross_library":
        file_order = [rid for rid in file_order if spectrum_id_to_ingest_lib.get(rid) != HOST_INGEST_LIB]
    return file_order[:MAX_REFS_PER_CANDIDATE]


cap_audit_rows = []
for row in host_pool.itertuples(index=False):
    key = (row.query_id, row.candidate_connectivity_key)
    classified = row_classified_refs[key]
    corrected = protocol_eligible(classified, "mirror_aware")[:MAX_REFS_PER_CANDIDATE]
    old = old_cap_selection(row.candidate_connectivity_key, row.query_id, "cross_library")
    full_ranked = [rid for rid, _ in classified]  # already compat-rank ordered (walk pool), good enough for rank lookup
    # compat_rank of each corrected-selection reference within the FULL ranked list (not just the walk pool)
    qmeta = query_meta.loc[row.query_id]
    full_reference_order = ranked_full_reference_list(row.candidate_connectivity_key, row.query_id,
                                                        qmeta["adduct"], qmeta["ce_mean"], qmeta["ionization_mode"], qmeta["instrument_type"])
    rank_of = {rid: i for i, rid in enumerate(full_reference_order)}
    cap_audit_rows.append({
        "query_id": row.query_id, "candidate_connectivity_key": row.candidate_connectivity_key,
        "is_true_candidate": bool(row.is_true_candidate),
        "n_full_references": len(full_reference_order),
        "old_selection": old, "corrected_selection": corrected,
        "selection_changed": set(old) != set(corrected),
        "max_compat_rank_selected": max((rank_of.get(rid, -1) for rid in corrected), default=-1),
    })
cap_audit_df = pd.DataFrame(cap_audit_rows)

frac_changed = cap_audit_df["selection_changed"].mean()
frac_changed_true = cap_audit_df.loc[cap_audit_df["is_true_candidate"], "selection_changed"].mean()
ranks_selected = cap_audit_df["max_compat_rank_selected"]
print(f"selection CHANGED (old vs corrected): {frac_changed:.2%} of all rows, {frac_changed_true:.2%} of true-candidate rows")
print(f"max compat_rank among selected references: median={ranks_selected.median():.0f}, P90={ranks_selected.quantile(0.9):.0f}, "
      f"P95={ranks_selected.quantile(0.95):.0f}, P99={ranks_selected.quantile(0.99):.0f}, max={ranks_selected.max()}")
print(f"fraction of rows where the corrected selection reached beyond compat_rank 30 (old cap): {(ranks_selected > 30).mean():.2%}")
print(f"fraction of rows where the corrected selection reached beyond compat_rank 50: {(ranks_selected > 50).mean():.2%}")

save_artifact(cap_audit_df.assign(old_selection=cap_audit_df["old_selection"].apply(str), corrected_selection=cap_audit_df["corrected_selection"].apply(str)),
              "host_reference_cap_audit", HOST_PROCESSED, description="Section 7 (v3): old file-order-capped vs corrected compat-rank selection, per row")
checks.append(CheckResult("old cap bug reproduced and audited against corrected selection", True, f"{frac_changed:.2%} rows changed"))

selection CHANGED (old vs corrected): 51.42% of all rows, 98.31% of true-candidate rows
max compat_rank among selected references: median=4, P90=4, P95=4, P99=4, max=10
fraction of rows where the corrected selection reached beyond compat_rank 30 (old cap): 0.00%
fraction of rows where the corrected selection reached beyond compat_rank 50: 0.00%


## 8. Section 8 — build pair features for all 4 protocols (pure filters over the shared classified walk pool)

In [10]:
t0 = time.time()
host_fold_by_query = host_holdout_spectra.set_index("query_id")["fold"].to_dict()
records = {p: [] for p in PROTOCOLS}

for row in host_pool.itertuples(index=False):
    q_offset = offset_of(row.query_id)
    if q_offset not in host_peaks_by_offset:
        continue
    q_peaks = host_peaks_by_offset[q_offset]
    key = (row.query_id, row.candidate_connectivity_key)
    classified = row_classified_refs[key]
    for protocol in PROTOCOLS:
        ref_ids = protocol_eligible(classified, protocol)[:MAX_REFS_PER_CANDIDATE]
        ref_peaks_list = [host_peaks_by_offset[offset_of(rid)] for rid in ref_ids if offset_of(rid) in host_peaks_by_offset]
        scores = compute_pair_spectral_scores(q_peaks, ref_peaks_list, bin_width=BIN_WIDTH_DA, peak_tol_da=PEAK_TOL_DA)
        record = {
            "query_id": row.query_id, "candidate_connectivity_key": row.candidate_connectivity_key,
            "abs_mass_error_ppm": row.abs_mass_error_ppm, "is_true_candidate": bool(row.is_true_candidate),
            "fold": host_fold_by_query[row.query_id],
        }
        record.update(scores)
        records[protocol].append(record)

host_pair_features = {p: pd.DataFrame.from_records(recs) for p, recs in records.items()}
print(f"built {len(PROTOCOLS)} protocol pair-feature tables in {time.time() - t0:.1f}s: " +
      ", ".join(f"{p}={len(df):,}" for p, df in host_pair_features.items()))
for name, df in host_pair_features.items():
    save_artifact(df, f"host_pair_features_{name}", HOST_INTERIM, description=f"Host holdout pair features (v3), {name} protocol")
checks.append(CheckResult("4 protocols built from shared classified pool", all(len(d) > 0 for d in host_pair_features.values()), ""))

built 4 protocol pair-feature tables in 543.4s: standard=126,573, cross_library=126,573, mirror_aware=126,573, near_dup_strict=126,573


## 9. Reproduce notebook 09's exact fold models (unchanged -- audit subject, not a new model)

In [11]:
pair_features_dev_known = pd.read_parquet(RANK_INTERIM / "pair_features_dev_known_spectrum.parquet")
dev_ranking_queries = pd.read_parquet(RANK_INTERIM / "dev_ranking_queries.parquet")
train_table = pair_features_dev_known.merge(dev_ranking_queries[["query_id", "fold", "true_connectivity_key"]], on="query_id", how="left")
assert_no_prohibited_features(train_table[MODEL_FEATURES], SPECTRAL_RANKING_FEATURE_POLICY)

fold_models = {}
for held_out_fold in sorted(train_table["fold"].unique()):
    tr = train_table[train_table["fold"] != held_out_fold].sort_values("query_id")
    tr_group = tr.groupby("query_id", sort=True).size().to_numpy()
    model = lgb.LGBMRanker(**LGBM_PARAMS)
    model.fit(tr[MODEL_FEATURES], tr["is_true_candidate"].astype(int), group=tr_group)
    fold_models[held_out_fold] = model

mode_a_mrr25_registered = load_promoted_baseline("notebook_09_lambdamart", BASELINE_REGISTRY_PATH, regime="mode_a")["mrr_at_25"]
print(f"reproduced {len(fold_models)} fold models (registered notebook 09 Mode A MRR@25 for reference: {mode_a_mrr25_registered:.4f})")
print("NOTE (blocking FROZEN): this is v0/historical -- trained on STANDARD-protocol DEV features,")
print("      evaluated here on mirror_aware-protocol HOST features. Protocol-matched retraining (V1-V4)")
print("      is explicitly deferred this notebook -- see freeze decision at the end.")
checks.append(CheckResult("fold models reproduced (v0, protocol-mismatched, diagnostic only)", len(fold_models) == 5, ""))

reproduced 5 fold models (registered notebook 09 Mode A MRR@25 for reference: 0.7531)
NOTE (blocking FROZEN): this is v0/historical -- trained on STANDARD-protocol DEV features,
      evaluated here on mirror_aware-protocol HOST features. Protocol-matched retraining (V1-V4)
      is explicitly deferred this notebook -- see freeze decision at the end.


## 10. Score each protocol: mass-only, peak-overlap, LambdaMART (v0, diagnostic)

In [12]:
def score_protocol(pair_df):
    lgbm_scores = pd.Series(np.nan, index=pair_df.index)
    for fold_value, model in fold_models.items():
        mask = pair_df["fold"] == fold_value
        if mask.any():
            lgbm_scores.loc[mask] = model.predict(pair_df.loc[mask, MODEL_FEATURES])
    assert lgbm_scores.notna().all()

    results = {}
    for method_name, series in [
        ("mass_only", -pair_df["abs_mass_error_ppm"]),
        ("peak_overlap_max", pair_df["peak_overlap_frac_max"]),
        ("lambdamart", lgbm_scores),
    ]:
        # a strict protocol (mirror_aware/near_dup_strict) can leave a candidate with ZERO
        # eligible references -> compute_pair_spectral_scores returns NaN for every spectral
        # field for that row (by design, so mass-only ranking stays possible) -- but a raw
        # classical score column (peak_overlap_max) used AS the ranking score directly can't
        # rank a NaN; drop those rows before ranking (matches notebook 09's own convention),
        # ranking_metrics' end-to-end reindex still counts the query as a miss (0), not dropped
        # from the denominator.
        scored = pair_df.assign(score=series).dropna(subset=["score"])
        ranked = rank_candidates(scored, score_col="score", ascending=False)
        metrics = ranking_metrics(ranked, all_host_query_ids, k_values=(1, 5, 10, 25))
        ranks = true_candidate_rank(ranked).reindex(all_host_query_ids)
        rr = per_query_reciprocal_rank(ranks, k=25)
        results[method_name] = {"metrics": metrics, "per_query_rr": rr}
    return results


protocol_results = {p: score_protocol(df) for p, df in host_pair_features.items()}
host_summary_rows = []
for protocol_name, methods in protocol_results.items():
    for method_name, r in methods.items():
        e2e = r["metrics"]["end_to_end"]
        host_summary_rows.append({"protocol": protocol_name, "method": method_name, "mrr_at_25": e2e["mrr_at_25"],
                                    "hit_at_1": e2e["hit_at_1"], "hit_at_5": e2e["hit_at_5"], "hit_at_25": e2e["hit_at_25"],
                                    "candidate_recall": e2e["candidate_recall"], "n_queries": e2e["n_queries"]})
host_mode_a_metrics = pd.DataFrame(host_summary_rows)
display(host_mode_a_metrics.pivot(index="method", columns="protocol", values="mrr_at_25"))
save_artifact(host_mode_a_metrics, "host_mode_a_metrics_v3", HOST_PROCESSED,
              description="v3: host Mode-A MRR@25/Hit@K under 4 protocols (standard/cross_library/mirror_aware/near_dup_strict) x 3 methods, v0 diagnostic ranker")
checks.append(CheckResult("host Mode-A scored under all 4 protocols x 3 methods", len(host_mode_a_metrics) == len(PROTOCOLS) * 3, ""))

protocol,cross_library,mirror_aware,near_dup_strict,standard
method,,,,
lambdamart,0.713386,0.713386,0.723297,0.799365
mass_only,0.399144,0.399144,0.399144,0.399144
peak_overlap_max,0.576034,0.576034,0.566549,0.726651


## 11. Paired bootstrap CIs (spectrum-level -- cluster/connectivity bootstrap explicitly deferred)

In [13]:
primary = protocol_results[PRIMARY_PROTOCOL]
bootstrap_rows = []
for a_name, b_name in [("lambdamart", "peak_overlap_max"), ("lambdamart", "mass_only"), ("peak_overlap_max", "mass_only")]:
    row = paired_bootstrap_report("reciprocal_rank@25", a_name, primary[a_name]["per_query_rr"], b_name, primary[b_name]["per_query_rr"],
                                   n_bootstrap=N_BOOTSTRAP, seed=SEED)
    row["protocol"] = PRIMARY_PROTOCOL
    bootstrap_rows.append(row)
paired_bootstrap_comparisons = pd.DataFrame(bootstrap_rows)
display(paired_bootstrap_comparisons[["protocol", "method_a", "method_b", "mean_a", "mean_b", "mean_diff", "ci_low", "ci_high", "significant_at_95"]])

lgbm_vs_peak_overlap = paired_bootstrap_comparisons.query("method_a == 'lambdamart' and method_b == 'peak_overlap_max'").iloc[0]
host_lgbm_mrr25 = host_mode_a_metrics.query("protocol == @PRIMARY_PROTOCOL and method == 'lambdamart'")["mrr_at_25"].iloc[0]
optimism_gap = mode_a_mrr25_registered - host_lgbm_mrr25
save_artifact(paired_bootstrap_comparisons, "paired_bootstrap_comparisons_v3", HOST_PROCESSED, description="v3 spectrum-level bootstrap, mirror_aware protocol (diagnostic v0 model)")
checks.append(CheckResult("paired bootstrap CIs computed (v0 model, mirror_aware protocol)", len(paired_bootstrap_comparisons) == 3, ""))

,protocol,method_a,method_b,mean_a,mean_b,mean_diff,ci_low,ci_high,significant_at_95
0,mirror_aware,lambdamart,peak_overlap_max,0.713386,0.576034,0.137352,0.117195,0.157760,True
1,mirror_aware,lambdamart,mass_only,0.713386,0.399144,0.314242,0.290120,0.337124,True
2,mirror_aware,peak_overlap_max,mass_only,0.576034,0.399144,0.176890,0.149487,0.202289,True


## 12. Section 6.2 — live symmetry check (complements `tests/test_deduplication.py`'s unit-level symmetry test)

In [14]:
# every classification row came from the SAME code path (classify_reference_relationship,
# called identically) regardless of is_true_candidate -- verify no candidate got a privileged
# path by checking relationship-rate parity isn't structurally impossible (a crude but real
# live check: decoys were in fact walked and classified at all, not skipped).
n_true_rows_classified = reference_classification.loc[reference_classification["is_true_candidate"], "query_id"].nunique()
n_decoy_rows_classified = (~reference_classification["is_true_candidate"]).sum()
symmetry_ok = n_true_rows_classified > 0 and n_decoy_rows_classified > 0
print(f"true-candidate rows with >=1 classified reference: {n_true_rows_classified}")
print(f"decoy (query, candidate, reference) triples classified: {n_decoy_rows_classified:,}")
print(f"symmetry check: {'PASS' if symmetry_ok else 'FAIL'} -- decoys were walked and classified through the identical code path, not skipped")
checks.append(CheckResult("decoy candidates walked/classified symmetrically with true candidates (live check)", symmetry_ok,
                           f"{n_decoy_rows_classified:,} decoy triples classified"))

true-candidate rows with >=1 classified reference: 1184
decoy (query, candidate, reference) triples classified: 882,925
symmetry check: PASS -- decoys were walked and classified through the identical code path, not skipped


## 13. Engineering checks (section 22) + scientific readiness (section 23)

In [15]:
engineering_checks = {
    "QCR-equivalent (classified walk pool) built and cached as artifacts": True,
    "no pre-filter reference cap anywhere (filter/rank happens on the FULL list first)": True,
    "reference ordering deterministic (compat_rank, no file order)": True,
    "eligibility/classification symmetric true/decoy": symmetry_ok,
    "old cap bug reproduced and audited": True,
    "deep-rank references measured (compat_rank distribution)": True,
    "all 4 protocols generated from the same shared classified pool": True,
    "DEV/HOST protocol code identical": False,  # DEV feature rebuild not done this notebook
    "cache invalidation fingerprints complete (proper artifact_fingerprint-based, not input_fingerprints={})": False,
    "warm rerun <=60s": False,  # not attempted -- no caching layer built for the QCR walk itself yet
}
scientific_readiness = {
    "host governance fixed (use_for_model_selection=False enforced)": True,
    "protocol-matched training (V1: mirror_aware DEV features)": False,
    "source-held-out variant (V2)": False,
    "source+structure-held-out variant (V3)": False,
    "mass likelihood model": False,
    "transparent mass+spectrum comparator (B1-B5)": False,
    "connectivity-cluster bootstrap": False,
    "candidate-density analysis (DEV/HOST/TEST)": False,
    "near-miss analysis": False,
    "reference-availability audit": True,
    "mirror vs near-duplicate decomposition": True,
}

eng_df = pd.DataFrame([{"check": k, "status": "PASS" if v else "PENDING"} for k, v in engineering_checks.items()])
sci_df = pd.DataFrame([{"item": k, "status": "DONE" if v else "PENDING"} for k, v in scientific_readiness.items()])
print("ENGINEERING CHECKS:")
display(eng_df)
print("\nSCIENTIFIC READINESS:")
display(sci_df)

n_eng_pending = (~eng_df["status"].eq("PASS")).sum()
n_sci_pending = (~sci_df["status"].eq("DONE")).sum()
checks.append(CheckResult("engineering checks computed", True, f"{n_eng_pending} pending"))
checks.append(CheckResult("scientific readiness checklist computed", True, f"{n_sci_pending} pending"))

ENGINEERING CHECKS:


,check,status
0,QCR-equivalent (classified walk pool) built an...,PASS
1,no pre-filter reference cap anywhere (filter/r...,PASS
2,"reference ordering deterministic (compat_rank,...",PASS
3,eligibility/classification symmetric true/decoy,PASS
4,old cap bug reproduced and audited,PASS
5,deep-rank references measured (compat_rank dis...,PASS
6,all 4 protocols generated from the same shared...,PASS
7,DEV/HOST protocol code identical,PENDING
8,cache invalidation fingerprints complete (prop...,PENDING
9,warm rerun <=60s,PENDING



SCIENTIFIC READINESS:


,item,status
0,host governance fixed (use_for_model_selection...,DONE
1,protocol-matched training (V1: mirror_aware DE...,PENDING
2,source-held-out variant (V2),PENDING
3,source+structure-held-out variant (V3),PENDING
4,mass likelihood model,PENDING
5,transparent mass+spectrum comparator (B1-B5),PENDING
6,connectivity-cluster bootstrap,PENDING
7,candidate-density analysis (DEV/HOST/TEST),PENDING
8,near-miss analysis,PENDING
9,reference-availability audit,DONE


## 14. Section 24 — Mode-A freeze decision (methodological completeness, NOT score-based per section 5.2)

In [16]:
blocking_items = [item for item, done in scientific_readiness.items() if not done] + \
                  [item for item, ok in engineering_checks.items() if not ok]
is_frozen = len(blocking_items) == 0
mode_a_status = "FROZEN" if is_frozen else "PROVISIONAL"

decision_log_entries = load_decision_log(DECISION_LOG_PATH)
post_host_changes = sum(1 for e in decision_log_entries if e["category"] == "post_host_change")

registry_entry = {
    "name": "host_mode_a_v3", "status": mode_a_status, "protocol": PRIMARY_PROTOCOL,
    "use_for_model_selection": False, "use_for_confirmation": True,
    "headline_variant": "v0_historical (protocol-mismatched, DIAGNOSTIC ONLY -- not a valid freeze candidate)",
    "dev_selection_metric": None, "host_confirmation_metric": "mrr_at_25",
    "host_mrr_at_25_diagnostic": float(host_lgbm_mrr25),
    "blocking_items": blocking_items,
    "post_host_decision_log_changes": post_host_changes,
    "artifact_hashes": {},
}
with open(REPORTS_DIR / "nb10_v3_summary.json", "w", encoding="utf-8") as f:
    json.dump(registry_entry, f, indent=2, default=str)

register_baseline("host_mode_a_v3", registry_entry, BASELINE_REGISTRY_PATH, source_notebook="10v3_host_mode_a_freeze.ipynb", overwrite=True)
with open(BASELINE_REGISTRY_PATH, encoding="utf-8") as f:
    full_registry = json.load(f)
with open(REPORTS_DIR / "baseline_registry.json", "w", encoding="utf-8") as f:
    json.dump(full_registry, f, indent=2, default=str)

print(f"MODE-A STATUS: {mode_a_status}")
if not is_frozen:
    print(f"\nBlocking items ({len(blocking_items)}):")
    for item in blocking_items:
        print(f"  - {item}")
checks.append(CheckResult("freeze decision computed and registered (methodological, not score-based)", True, f"status={mode_a_status}, {len(blocking_items)} blocking items"))

MODE-A STATUS: PROVISIONAL

Blocking items (11):
  - protocol-matched training (V1: mirror_aware DEV features)
  - source-held-out variant (V2)
  - source+structure-held-out variant (V3)
  - mass likelihood model
  - transparent mass+spectrum comparator (B1-B5)
  - connectivity-cluster bootstrap
  - candidate-density analysis (DEV/HOST/TEST)
  - near-miss analysis
  - DEV/HOST protocol code identical
  - cache invalidation fingerprints complete (proper artifact_fingerprint-based, not input_fingerprints={})
  - warm rerun <=60s


## 15. Final validation checklist + Task-1 summary report

In [17]:
all_passed, checks_dict = summarize_checks(checks)
checklist_df = pd.DataFrame([{"Check": name, "Status": "PASS" if ok else "FAIL"} for name, ok in checks_dict.items()])
display(checklist_df)

TOTAL_RUNTIME_S = time.time() - NOTEBOOK_START_TIME
print("=" * 70)
print("NOTEBOOK 10 v3 -- MODE-A FREEZE REPORT")
print("=" * 70)
print(f"\nHost: {len(host_holdout_spectra):,} spectra, {host_holdout_spectra['true_connectivity_key'].nunique():,} connectivities, "
      f"instrument(s)={sorted(host_holdout_spectra['instrument_type'].unique())}")
primary_true_coverage = reference_coverage.query("protocol == @PRIMARY_PROTOCOL and group == 'true_candidates'")["frac_ge1_ref"].iloc[0]
print(f"\nReference audit:")
print(f"    true candidates with >=1 eligible ref ({PRIMARY_PROTOCOL}): {primary_true_coverage:.2%}")
print(f"    cross-source mirrors: {n_cross_source_mirror} ({queries_with_cross_source_mirror} queries)")
print(f"    cross-source near-duplicates (non-mirror): {n_cross_source_near_dup} ({queries_with_cross_source_near_dup} queries)")
print(f"    decoy-candidate mirrors (symmetry check): {n_decoy_mirrors}")
print(f"    selected refs with compat_rank > 50 (old-cap-bug relevance): {(ranks_selected > 50).mean():.2%}")
print(f"\nDiagnostic (v0, protocol-MISMATCHED) ranker:")
print(f"    HOST {PRIMARY_PROTOCOL} MRR@25: {host_lgbm_mrr25:.4f} (vs DEV in-sample {mode_a_mrr25_registered:.4f}, optimism gap {optimism_gap:+.4f})")
print(f"    LambdaMART vs peak_overlap_max: diff={lgbm_vs_peak_overlap['mean_diff']:+.4f} "
      f"[{lgbm_vs_peak_overlap['ci_low']:+.4f}, {lgbm_vs_peak_overlap['ci_high']:+.4f}] "
      f"({'SIGNIFICANT' if lgbm_vs_peak_overlap['significant_at_95'] else 'not significant'}, spectrum-level CI)")
print(f"\nHeadline variant: NONE SELECTED -- protocol-matched V1-V4 retraining not done this notebook (see blocking items)")
print(f"\nWarm runtime: not measured (no dedicated cache layer built for the walk/classification step yet)")
print(f"\nMODE-A STATUS: {mode_a_status}")
print(f"\nBlocking items ({len(blocking_items)}):")
for item in blocking_items:
    print(f"    - {item}")
print(f"\nRuntime this run: {TOTAL_RUNTIME_S:.1f}s")
print("=" * 70)
print(f"\n{'ALL CHECKS PASS' if all_passed else 'SOME CHECKS FAILED'} -- {'TASK 2 (Notebook 11) MAY NOT START' if not is_frozen else 'Task 2 may proceed'}")

,Check,Status
0,host holdout + candidate pool reloaded,PASS
1,deterministic compat-rank reference ordering i...,PASS
2,"symmetric walk pool built for every (query, ca...",PASS
3,host peak arrays loaded (symmetric walk pool),PASS
4,symmetric mirror/near-duplicate classification...,PASS
5,"mirror/near-duplicate breakdown computed, symm...",PASS
6,"reference availability audit computed, true/de...",PASS
7,old cap bug reproduced and audited against cor...,PASS
8,4 protocols built from shared classified pool,PASS
9,"fold models reproduced (v0, protocol-mismatche...",PASS


NOTEBOOK 10 v3 -- MODE-A FREEZE REPORT

Host: 1,184 spectra, 250 connectivities, instrument(s)=['timsTOF']

Reference audit:
    true candidates with >=1 eligible ref (mirror_aware): 100.00%
    cross-source mirrors: 0 (0 queries)
    cross-source near-duplicates (non-mirror): 2683 (423 queries)
    decoy-candidate mirrors (symmetry check): 0
    selected refs with compat_rank > 50 (old-cap-bug relevance): 0.00%

Diagnostic (v0, protocol-MISMATCHED) ranker:
    HOST mirror_aware MRR@25: 0.7134 (vs DEV in-sample 0.7531, optimism gap +0.0397)
    LambdaMART vs peak_overlap_max: diff=+0.1374 [+0.1172, +0.1578] (SIGNIFICANT, spectrum-level CI)

Headline variant: NONE SELECTED -- protocol-matched V1-V4 retraining not done this notebook (see blocking items)

Warm runtime: not measured (no dedicated cache layer built for the walk/classification step yet)

MODE-A STATUS: PROVISIONAL

Blocking items (11):
    - protocol-matched training (V1: mirror_aware DEV features)
    - source-held-out vari